In [1]:
import os
import kagglehub

print("Downloading PACS dataset via kagglehub...")
path = kagglehub.dataset_download("ma3ple/pacs-dataset")

print("Path to dataset files:", path)

current_path = path
while 'art_painting' not in [d.lower() for d in os.listdir(current_path)]:
    subdirs = [d for d in os.listdir(current_path) if os.path.isdir(os.path.join(current_path, d))]
    if not subdirs:
        break
    current_path = os.path.join(current_path, subdirs[0])

DATA_ROOT = current_path
print(f"Data Path: {DATA_ROOT}")
print(f"Domain: {os.listdir(DATA_ROOT)}")

c:\Users\minky\ml-intern-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\minky\.cache\kagglehub\datasets\ma3ple\pacs-dataset\versions\1
Data Path: C:\Users\minky\.cache\kagglehub\datasets\ma3ple\pacs-dataset\versions\1\kfold
Domain: ['art_painting', 'cartoon', 'photo', 'sketch']


In [ ]:
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, Subset
from PIL import Image, ImageFilter

import cv2

# ===================== 설정 =====================
BATCH_SIZE = 10
LR = 5e-4
EPOCHS = 35           
WEIGHT_DECAY = 1e-2    # AdamW 기준. SGD였다면 5e-4
VAL_FRAC = 0.12        # photo에서 검증용으로 떼는 비율
EDGE_P = 0.75           # 학습 중 엣지 증강을 적용할 확률
GRAY_P = 1.0           # 학습 중 흑백 변환 확률
NUM_WORKERS = 0        # Windows+Jupyter에서 커스텀 transform은 워커>0이면 pickle 실패

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

data_root = DATA_ROOT
while 'kfold' in os.listdir(data_root):
    data_root = os.path.join(data_root, 'kfold')

print(f"Actual Data Root: {data_root}")
print(f"Found Domains: {os.listdir(data_root)}")


# ===================== 엣지 변환 =====================
class RandomEdge:
    """학습용 엣지 증강. 선 스타일 2종을 섞어 한 연산자에 과적합되는 것을 막는다.
    임계값은 진짜 sketch의 잉크 비율(4.4%)에 맞춰 실측으로 잡았다.
    (기존 Canny t1 in [50,150] + dilate 는 잉크 18.8%로 진짜보다 4배 진했다)"""

    def __init__(self, p=0.4):
        self.p = p

    def __call__(self, img):
        if random.random() > self.p:
            return img

        gray = np.array(img.convert("L"))

        if random.random() < 0.5:
            # Canny: 얇고 또렷한 선
            t1 = random.randint(200, 300)
            out = 255 - cv2.Canny(gray, t1, t1 + random.randint(60, 180))
        else:
            # XDoG: 연필 스케치풍. 굵기와 농담이 자연스럽다
            g = gray.astype(np.float32) / 255.0
            s = random.uniform(0.6, 1.4)
            d = cv2.GaussianBlur(g, (0, 0), s) - 0.98 * cv2.GaussianBlur(g, (0, 0), s * 1.6)
            eps = random.uniform(-0.009, -0.003)
            soft = np.where(d >= eps, 1.0, 1.0 + np.tanh(random.uniform(15, 40) * (d - eps)))
            out = (np.clip(soft, 0, 1) * 255).astype(np.uint8)

        return Image.fromarray(out).convert("RGB")


class FixedSketchify:
    """검증용 가짜 sketch. 랜덤 없음, 전부 적용, 학습과 다른 연산자를 쓴다."""

    def __call__(self, img):
        e = img.convert("L").filter(ImageFilter.CONTOUR)       # 흰 배경 + 검은 선
        return e.convert("RGB")


# ===================== transform 3종 =====================
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4)], p=1.0),
    transforms.RandomGrayscale(p=GRAY_P),
    RandomEdge(p=EDGE_P),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

eval_transform = transforms.Compose([          # 증강 없음. 평가는 항상 결정적으로
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

pseudo_sketch_transform = transforms.Compose([  # 크기 먼저, 그 다음 엣지
    transforms.Resize((224, 224)),
    FixedSketchify(),
    transforms.ToTensor(),
    transforms.Normalize(mean=MEAN, std=STD),
])

# ===================== 데이터셋 =====================
all_domain_folders = os.listdir(data_root)
train_domains = ['photo']
test_domain = 'sketch'

photo_dir = os.path.join(data_root, [f for f in all_domain_folders if 'photo' in f.lower()][0])
test_dir = os.path.join(data_root, [f for f in all_domain_folders if test_domain in f.lower()][0])

# 같은 폴더를 transform만 바꿔서 세 번 연다
photo_aug = datasets.ImageFolder(photo_dir, transform=train_transform)
photo_plain = datasets.ImageFolder(photo_dir, transform=eval_transform)
photo_pseudo = datasets.ImageFolder(photo_dir, transform=pseudo_sketch_transform)

test_dataset = datasets.ImageFolder(test_dir, transform=eval_transform)
assert photo_aug.classes == test_dataset.classes, "photo와 sketch의 클래스 순서가 다름"

# --- stratified split: 클래스마다 VAL_FRAC 만큼씩 떼어낸다 ---
targets = np.array(photo_aug.targets)
rng = np.random.RandomState()
train_idx, val_idx = [], []
for c in np.unique(targets):
    idx_c = np.where(targets == c)[0]
    rng.shuffle(idx_c)
    n_val = int(round(len(idx_c) * VAL_FRAC))
    val_idx.extend(idx_c[:n_val].tolist())
    train_idx.extend(idx_c[n_val:].tolist())

train_set = Subset(photo_aug, train_idx)         # 증강 O
val_photo_set = Subset(photo_plain, val_idx)     # 증강 X, 원본 photo
val_sketch_set = Subset(photo_pseudo, val_idx)   # 증강 X, 고정 엣지 변환

_kw = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
train_loader = DataLoader(train_set, shuffle=True, **_kw)
val_photo_loader = DataLoader(val_photo_set, shuffle=False, **_kw)
val_sketch_loader = DataLoader(val_sketch_set, shuffle=False, **_kw)
test_loader = DataLoader(test_dataset, shuffle=False, **_kw)
print("\n클래스별 분포 (train / val):")
for c, name in enumerate(photo_aug.classes):
    n_tr = int((targets[train_idx] == c).sum())
    n_va = int((targets[val_idx] == c).sum())
    print(f"  {name:<12} {n_tr:>4} / {n_va:>3}")

In [ ]:
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn

# ResNet-18 (scratch. 규정상 weights=None 고정)
model = models.resnet18(weights=None)

# (dog, elephant, giraffe, guitar, horse, house, person)
model.fc = nn.Linear(model.fc.in_features, 7)
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# --- AdamW: weight decay를 conv/fc 가중치에만 적용 ---
# BatchNorm의 gamma/beta와 bias는 1차원이며, 0으로 끌어당기면 오히려 해가 된다.
decay_params, no_decay_params = [], []
for name, p in model.named_parameters():
    if not p.requires_grad:
        continue
    (no_decay_params if p.ndim <= 1 else decay_params).append(p)

optimizer = optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=LR,
)

# --- Cosine annealing: LR을 EPOCHS에 걸쳐 LR -> 0 으로 부드럽게 줄인다 ---
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-4)

# --- Polyak averaging (EMA) ---
# 감쇠율을 직접 쓰지 않고 "몇 epoch 분량을 평균할지"에서 역산한다.
# 데이터가 1,420장뿐이라 epoch당 스텝이 12번 정도밖에 안 된다.
# 흔히 쓰는 0.999를 그대로 넣으면 유효 창이 1000스텝이라 학습 전체보다 길어져
# EMA 모델이 초기 랜덤 가중치에 머무는 사고가 난다.
EMA_WINDOW_EPOCHS = 3
steps_per_epoch = len(train_loader)
EMA_DECAY = 1.0 - 1.0 / (EMA_WINDOW_EPOCHS * steps_per_epoch)

ema_model = AveragedModel(
    model,
    multi_avg_fn=get_ema_multi_avg_fn(EMA_DECAY),
    use_buffers=True,          # BatchNorm 통계(running_mean/var)도 같이 평균
)

n_decay = sum(p.numel() for p in decay_params)
n_no_decay = sum(p.numel() for p in no_decay_params)
print(f"weight decay 적용: {n_decay:,}개 / 제외(BN·bias): {n_no_decay:,}개")
print(f"optimizer: AdamW(lr={LR}, weight_decay={WEIGHT_DECAY})")
print(f"scheduler: CosineAnnealingLR(T_max={EPOCHS})  {LR:.1e} -> 0")
print(f"EMA decay = {EMA_DECAY:.4f} "
      f"({steps_per_epoch} steps/epoch x {EMA_WINDOW_EPOCHS} epochs = {EMA_WINDOW_EPOCHS*steps_per_epoch} steps 창)")

weight decay 적용: 11,170,496개 / 제외(BN·bias): 9,607개
optimizer: AdamW(lr=0.0005, weight_decay=0.01)
scheduler: CosineAnnealingLR(T_max=35)  5.0e-04 -> 0
EMA decay = 0.9977 (147 steps/epoch x 3 epochs = 441 steps 창)


In [ ]:
import time


@torch.no_grad()
def run_eval(net, loader):
    net.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = net(images)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return 100. * correct / total


def train_one_epoch(epoch):
    t0 = time.time()
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        ema_model.update_parameters(model)     # Polyak: 매 스텝마다 평균 갱신

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    lr_now = optimizer.param_groups[0]["lr"]
    scheduler.step()                            # Cosine: epoch 끝날 때 LR 갱신

    # 검증은 photo에서 떼어둔 것만 사용한다. sketch는 여기서 절대 보지 않는다.
    val_photo = run_eval(model, val_photo_loader)
    val_pseudo = run_eval(model, val_sketch_loader)
    ema_pseudo = run_eval(ema_model, val_sketch_loader)

    print(f"Epoch {epoch+1:>3} | Loss {running_loss/len(train_loader):.4f} "
          f"| Train {100.*correct/total:5.2f}% "
          f"| ValPhoto {val_photo:5.2f}% | ValSketch {val_pseudo:5.2f}% "
          f"| EMA {ema_pseudo:5.2f}% | lr {lr_now:.2e} | {time.time()-t0:5.1f}s")

    return val_pseudo, ema_pseudo


print("Challenge Start!")
total_t0 = time.time()
history = []
for epoch in range(EPOCHS):
    history.append(train_one_epoch(epoch))
print(f"Total training time: {time.time() - total_t0:.1f}s")

# ===== 여기서부터가 최종 채점. sketch를 처음이자 마지막으로 사용한다 =====
print("\n" + "=" * 62)
raw_acc = run_eval(model, test_loader)
ema_acc = run_eval(ema_model, test_loader)
print(f"[OOD Evaluation] Target Domain (Sketch) Accuracy : {raw_acc:.2f}%")
print(f"[OOD Evaluation] Polyak/EMA averaged model       : {ema_acc:.2f}%")
print("=" * 62)

final_acc = max(raw_acc, ema_acc)

Challenge Start!
Epoch   1 | Loss 1.9739 | Train 28.32% | ValPhoto 37.31% | ValSketch 42.29% | EMA 10.95% | lr 5.00e-04 |   9.7s
Epoch   2 | Loss 1.7909 | Train 35.40% | ValPhoto 27.36% | ValSketch 35.32% | EMA 25.87% | lr 4.99e-04 |   9.3s
Epoch   3 | Loss 1.6888 | Train 40.03% | ValPhoto 34.83% | ValSketch 54.73% | EMA 26.87% | lr 4.97e-04 |   9.3s
Epoch   4 | Loss 1.5917 | Train 44.59% | ValPhoto 43.78% | ValSketch 31.34% | EMA 30.85% | lr 4.93e-04 |   9.4s
Epoch   5 | Loss 1.4794 | Train 49.29% | ValPhoto 47.26% | ValSketch 36.82% | EMA 51.24% | lr 4.87e-04 |  10.4s
Epoch   6 | Loss 1.4239 | Train 52.48% | ValPhoto 39.80% | ValSketch 28.36% | EMA 58.71% | lr 4.80e-04 |   9.3s
Epoch   7 | Loss 1.3903 | Train 55.00% | ValPhoto 46.77% | ValSketch 53.23% | EMA 54.73% | lr 4.72e-04 |   9.4s
Epoch   8 | Loss 1.3452 | Train 56.64% | ValPhoto 52.24% | ValSketch 34.33% | EMA 59.20% | lr 4.62e-04 |   9.4s
Epoch   9 | Loss 1.3092 | Train 59.22% | ValPhoto 48.26% | ValSketch 58.21% | EMA 58.21